# 03 · Train (Kaggle GPU, T4 x2)

Each GPU works through its own queue of experiments, one after another. Checkpoints are uploaded to a private HF model repo every 30 min, so re-running this notebook with the same queues resumes unfinished runs and skips finished ones. A copy of every run also stays in `/kaggle/working/runs`, i.e. in this version's Output.

**Setup:** Settings → Accelerator: *GPU T4 x2*, Internet: *On*. Add-ons → Secrets: `HF_TOKEN` (write access), enabled for this notebook. Run with *Save Version → Save & Run All (Commit)*; no cell needs input.

**If uploads failed in a previous session** (its log shows `upload failed`): attach that version's Output with *Add Input*, set `RESTORE_FROM` to its `runs` folder and run again. Its checkpoints are uploaded to the Hub first, then training resumes from them.

| Session | GPU 0 | GPU 1 | ~Hours |
|---|---|---|---|
| 1 | tr000 → tr010 | tr100 → tr050 | 8.5 |
| 2 | tr030 | tr_only_control → tr030_seed1 | 5.5 |

In [ ]:
QUEUES = [
    ["tr000", "tr010"],  # GPU 0
    ["tr100", "tr050"],  # GPU 1
]
RUN_BENCHMARK = False  # 100-step throughput test before training (~12 min)
RESTORE_FROM = None  # e.g. "/kaggle/input/<previous-version-output>/runs"

HF_USER = "DeerKaraca"
DATA_REPO = f"{HF_USER}/bilingual-slm-data"
RUNS_REPO = f"{HF_USER}/bilingual-slm-runs"
WORK = "/tmp/slm"  # training data: large and re-downloadable, so not kept in the Output
RUNS = "/kaggle/working/runs"  # checkpoints: kept in the Output as a backup of the Hub copy

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Pinned to a tag: every run, in every session, uses the same training code.
!git clone -q --branch train-v1 https://github.com/DeerKaraca/bilingual-slm.git
%cd bilingual-slm
!git log --oneline -1
!pip install -q -e ".[data,train]"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, snapshot_download

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

# Fail now, not hours later: training only warns when an upload fails, so check that the token can write.
api = HfApi()
api.create_repo(RUNS_REPO, private=True, exist_ok=True)
api.upload_file(path_or_fileobj=b"ok", path_in_repo="token_check.txt", repo_id=RUNS_REPO)
print("HF_TOKEN can write to", RUNS_REPO)

snapshot_download(DATA_REPO, repo_type="dataset", allow_patterns=["tokens/*", "tokenizer.json"], local_dir=f"{WORK}/data")

In [ ]:
import torch
from pathlib import Path

# Training downloads latest.pt from the Hub over any local copy, so a backup from an earlier
# Output is restored by uploading it to the Hub first.
if RESTORE_FROM:
    for run_dir in sorted(Path(RESTORE_FROM).iterdir()):
        if (run_dir / "latest.pt").exists():
            step = torch.load(run_dir / "latest.pt", map_location="cpu", weights_only=False)["step"]
            for name in ("latest.pt", "log.csv"):
                api.upload_file(path_or_fileobj=run_dir / name, path_in_repo=f"{run_dir.name}/{name}", repo_id=RUNS_REPO)
            print(f"restored {run_dir.name} at step {step}")

In [ ]:
import subprocess
import sys


def train_cmd(name, *overrides):
    return [sys.executable, "-m", "slm.train", "--config", f"configs/experiments/{name}.yaml", "--set",
            f"data.tokens_dir={WORK}/data/tokens", f"data.tokenizer={WORK}/data/tokenizer.json", *overrides]


if RUN_BENCHMARK:
    subprocess.run(train_cmd("tr050", f"out_dir={WORK}/bench", "train.total_tokens=26214400", "train.eval_every=50"), check=True)

In [ ]:
import threading

LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(exist_ok=True)
exit_codes = {}


def run_queue(gpu, names):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    for name in names:
        with open(LOGS / f"{name}.log", "a") as log:
            cmd = train_cmd(name, f"out_dir={RUNS}", f"hub_repo={RUNS_REPO}")
            exit_codes[name] = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, env=env).returncode


threads = [threading.Thread(target=run_queue, args=(gpu, names)) for gpu, names in enumerate(QUEUES)]
for t in threads:
    t.start()

In [ ]:
import time

# Keep the session alive and print the latest log line of each run every 10 minutes.
while any(t.is_alive() for t in threads):
    time.sleep(600)
    for log in sorted(LOGS.glob("*.log")):
        text = log.read_text()
        lines = text.strip().splitlines()
        print(f"{log.stem}: {lines[-1][:200] if lines else '(starting)'}", flush=True)
        if "upload failed" in text:
            print(f"WARNING {log.stem}: checkpoint uploads are failing; the newest checkpoint is only in {RUNS}", flush=True)

print("exit codes:", exit_codes)
assert all(code == 0 for code in exit_codes.values()), "a run failed, see /kaggle/working/logs"